# 📖 Notebook 1: GPS Activity Tracking & Storage

In this notebook we build the core of a Strava-like fitness app: **recording GPS activities**,
calculating distance, and using PostGIS for spatial queries.

## Learning Objectives

By the end of this notebook, you'll understand:
- How GPS coordinates are recorded and stored during a run or ride
- The **Haversine formula** for calculating distance between two points on Earth
- How PostGIS extends Postgres with geographic superpowers
- How to handle **pause/resume** with a state log for accurate elapsed time
- Why Strava tracks data **locally on the device** and uploads on completion

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/strava
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `strava_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import math
import time
import json

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "strava_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

# Test the connection
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT PostGIS_Version();")
    version = cur.fetchone()[0]
    conn.close()
    print(f"✅ Connected to PostgreSQL with PostGIS {version}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🗄️ Exploring the Schema

Our database has these tables (created by `db/init.sql`):

| Table | Purpose |
|-------|---------|
| `users` | Athletes with a username and city |
| `activities` | Each run or ride — with state, distance, duration |
| `route_points` | GPS breadcrumbs (lat, lon, elevation, timestamp) |
| `activity_state_log` | Start/Pause/Resume events for accurate timing |
| `segments` | Famous stretches to race on |
| `segment_efforts` | One row per athlete per segment attempt |
| `friends` | Bi-directional friendship pairs |

Let's look at the data that was seeded for us.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# How many users, activities, and route points do we have?
for table in ['users', 'activities', 'route_points', 'segments']:
    cur.execute(f"SELECT COUNT(*) as cnt FROM {table}")
    print(f"  {table}: {cur.fetchone()['cnt']} rows")

print()
print("Sample activities:")
cur.execute("""
    SELECT a.id, u.username, a.type, a.state, a.title,
           ROUND(a.distance_m::numeric) as distance_m, a.duration_s
    FROM activities a
    JOIN users u ON a.user_id = u.id
    ORDER BY a.id
    LIMIT 8
""")
for row in cur.fetchall():
    print(f"  #{row['id']} {row['username']:8s} {row['type']:5s} {row['state']:10s} "
          f"{row['distance_m']:>7} m  {row['duration_s']:>5} s  {row['title']}")

conn.close()

## 🌍 How GPS Tracking Works

When you start a run on Strava, your phone records your position every few seconds:

```
Time 0s:  (37.7955, -122.3935)   ← you're at the Ferry Building
Time 5s:  (37.7950, -122.3928)   ← you moved south-east
Time 10s: (37.7944, -122.3919)   ← still going
...
```

Each point is a **(latitude, longitude)** pair — two numbers that pinpoint any location on Earth.

- **Latitude** ranges from -90 (South Pole) to +90 (North Pole)
- **Longitude** ranges from -180 (west) to +180 (east)

Let's look at Alice's run along the Embarcadero.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Fetch Alice's route points for activity #1
cur.execute("""
    SELECT seq, latitude, longitude, elevation_m, recorded_at
    FROM route_points
    WHERE activity_id = 1
    ORDER BY seq
""")
points = cur.fetchall()
conn.close()

print(f"Alice's Embarcadero run — {len(points)} GPS points:")
print(f"{'Seq':>4} {'Latitude':>10} {'Longitude':>12} {'Elev(m)':>8}")
print("-" * 40)
for p in points:
    print(f"{p['seq']:>4} {p['latitude']:>10.4f} {p['longitude']:>12.4f} {p['elevation_m']:>8.1f}")

print()
print("💡 Each row is a GPS 'breadcrumb' recorded every ~5 seconds.")
print("   Connect the dots and you get the route on a map!")

## 📏 The Haversine Formula: Distance on a Sphere

Earth isn't flat, so we can't just use the Pythagorean theorem on latitude/longitude.
Instead, we use the **Haversine formula** which accounts for Earth's curvature.

Think of it this way:
- Latitude and longitude are *angles* on a sphere
- The Haversine formula converts the angle difference into an arc length
- It gives us the **"as the crow flies"** distance between two points

The formula:
```
a = sin²(Δlat/2) + cos(lat1) · cos(lat2) · sin²(Δlon/2)
c = 2 · atan2(√a, √(1−a))
d = R · c          where R = 6,371,000 metres (Earth's radius)
```

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the distance in metres between two GPS points.
    This is the same formula Strava uses under the hood.
    """
    R = 6_371_000  # Earth's radius in metres

    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat / 2) ** 2 + \
        math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

# Test: distance between first two points of Alice's run
d = haversine(37.7955, -122.3935, 37.7950, -122.3928)
print(f"Distance between points 1 and 2: {d:.1f} metres")
print()

# Calculate total route distance by summing consecutive segments
total = 0
print(f"{'Seg':>4} {'From':>6} {'To':>4} {'Segment (m)':>12} {'Cumulative (m)':>16}")
print("-" * 50)
for i in range(len(points) - 1):
    seg_dist = haversine(
        points[i]['latitude'], points[i]['longitude'],
        points[i+1]['latitude'], points[i+1]['longitude']
    )
    total += seg_dist
    print(f"{i+1:>4} {points[i]['seq']:>6} → {points[i+1]['seq']:>2} {seg_dist:>12.1f} {total:>16.1f}")

print(f"\n📏 Total route distance: {total:.0f} metres ({total/1000:.2f} km)")
print(f"   Stored in database:   2540 metres")
print("\n💡 The small difference is because our seed data uses round numbers.")

## 🗺️ PostGIS: Spatial Superpowers for Postgres

PostGIS adds geographic data types and functions to Postgres. Instead of writing
Haversine by hand, PostGIS can do it **inside the database** — and it's indexed!

Our `route_points` table has a `geom` column of type `GEOGRAPHY(Point, 4326)`:
- `GEOGRAPHY` = uses real Earth coordinates (not flat-plane)
- `Point` = a single lat/lon location
- `4326` = WGS-84, the standard GPS coordinate system

Let's see what PostGIS can do.

In [ ]:
conn = get_db()
cur = conn.cursor()

# --- Example 1: Distance between two points using PostGIS ---
cur.execute("""
    SELECT ST_Distance(
        ST_Point(-122.3935, 37.7955)::geography,
        ST_Point(-122.3928, 37.7950)::geography
    ) AS distance_m;
""")
postgis_dist = cur.fetchone()[0]
print(f"PostGIS distance between points 1→2: {postgis_dist:.1f} m")
print(f"Our Haversine calculation:           {d:.1f} m")
print(f"Difference:                          {abs(postgis_dist - d):.2f} m")
print()
print("💡 They match! PostGIS uses the same math but inside the database.")

In [ ]:
# --- Example 2: Total route length using PostGIS ---
# Sum the distance between consecutive GPS points for an activity
cur.execute("""
    WITH ordered AS (
        SELECT geom,
               LAG(geom) OVER (ORDER BY seq) AS prev_geom
        FROM route_points
        WHERE activity_id = 1
    )
    SELECT ROUND(SUM(ST_Distance(geom, prev_geom))::numeric, 1) AS total_m
    FROM ordered
    WHERE prev_geom IS NOT NULL;
""")
postgis_total = cur.fetchone()[0]
print(f"PostGIS total route distance: {postgis_total} m")
print(f"Our Python calculation:       {total:.1f} m")
print()
print("💡 Same answer — but PostGIS does it in a single SQL query!")

In [ ]:
# --- Example 3: Find all route points within 500 m of a landmark ---
# The Ferry Building in SF is at (37.7956, -122.3933)
cur.execute("""
    SELECT rp.activity_id, a.title, rp.seq,
           ROUND(ST_Distance(
               rp.geom,
               ST_Point(-122.3933, 37.7956)::geography
           )::numeric) AS dist_m
    FROM route_points rp
    JOIN activities a ON rp.activity_id = a.id
    WHERE ST_DWithin(
        rp.geom,
        ST_Point(-122.3933, 37.7956)::geography,
        500  -- within 500 metres
    )
    ORDER BY dist_m
    LIMIT 10;
""")
rows = cur.fetchall()
print("Route points within 500 m of the SF Ferry Building:")
print(f"{'Activity':>10} {'Title':<30} {'Point#':>7} {'Distance':>10}")
print("-" * 62)
for r in rows:
    print(f"{r[0]:>10} {r[1]:<30} {r[2]:>7} {r[3]:>8} m")

print()
print("💡 ST_DWithin uses the spatial index — it's fast even with millions of points!")
conn.close()

## ⏸️ Pause/Resume: Accurate Elapsed Time

Athletes often pause mid-activity (red light, rest stop, shoe tying). We need to
track **active time** accurately, excluding paused periods.

The trick: maintain a **state log** — a list of timestamped events:

```
STARTED  at  8:00:00
PAUSED   at  8:10:00   ← 10 min of running
RESUMED  at  8:15:00   ← 5 min of rest (excluded)
COMPLETE at  8:25:00   ← 10 more min of running
                         Total active time = 20 min
```

Let's simulate this.

In [ ]:
from datetime import datetime, timedelta

def calculate_active_seconds(state_log):
    """
    Given a list of (state, timestamp) pairs, calculate total active seconds.
    Active time = time spent in STARTED or RESUMED state (not PAUSED).
    """
    active_seconds = 0
    active_since = None

    for state, ts in state_log:
        if state in ('STARTED', 'RESUMED'):
            active_since = ts  # clock starts
        elif state in ('PAUSED', 'COMPLETE'):
            if active_since:
                active_seconds += (ts - active_since).total_seconds()
                active_since = None

    return int(active_seconds)

# Simulate a run with a 5-minute pause
base = datetime(2025, 3, 15, 8, 0, 0)
log = [
    ('STARTED',  base),
    ('PAUSED',   base + timedelta(minutes=10)),
    ('RESUMED',  base + timedelta(minutes=15)),
    ('COMPLETE', base + timedelta(minutes=25)),
]

active = calculate_active_seconds(log)
wall_clock = (log[-1][1] - log[0][1]).total_seconds()

print("State Log:")
for state, ts in log:
    print(f"  {state:<10} at {ts.strftime('%H:%M:%S')}")
print()
print(f"Wall clock time: {wall_clock:.0f} seconds ({wall_clock/60:.0f} min)")
print(f"Active time:     {active} seconds ({active/60:.0f} min)")
print(f"Paused time:     {wall_clock - active:.0f} seconds ({(wall_clock-active)/60:.0f} min)")
print()
print("💡 Strava shows both 'Elapsed Time' and 'Moving Time' — this is how.")

In [ ]:
# Let's do this in the database with our state log table
conn = get_db()
cur = conn.cursor()

# Get Alice's in-progress activity (id=8)
activity_id = 8

# Insert a realistic state log
cur.execute("""
    DELETE FROM activity_state_log WHERE activity_id = %s;
    INSERT INTO activity_state_log (activity_id, state, recorded_at) VALUES
        (%s, 'STARTED',  NOW() - INTERVAL '30 minutes'),
        (%s, 'PAUSED',   NOW() - INTERVAL '20 minutes'),
        (%s, 'RESUMED',  NOW() - INTERVAL '15 minutes'),
        (%s, 'PAUSED',   NOW() - INTERVAL '5 minutes'),
        (%s, 'RESUMED',  NOW() - INTERVAL '3 minutes'),
        (%s, 'COMPLETE', NOW());
""", (activity_id,) + (activity_id,) * 6)

# Query to calculate active time from the state log
cur.execute("""
    WITH events AS (
        SELECT state, recorded_at,
               LEAD(recorded_at) OVER (ORDER BY recorded_at) AS next_at
        FROM activity_state_log
        WHERE activity_id = %s
    )
    SELECT
        EXTRACT(EPOCH FROM SUM(
            CASE WHEN state IN ('STARTED', 'RESUMED')
                 THEN next_at - recorded_at
            END
        ))::int AS active_seconds,
        EXTRACT(EPOCH FROM SUM(
            CASE WHEN state = 'PAUSED'
                 THEN next_at - recorded_at
            END
        ))::int AS paused_seconds
    FROM events
    WHERE next_at IS NOT NULL;
""", (activity_id,))

row = cur.fetchone()
print(f"Activity #{activity_id} timing (calculated from state log):")
print(f"  Active time: {row[0]} seconds ({row[0]//60} min)")
print(f"  Paused time: {row[1]} seconds ({row[1]//60} min)")
print(f"  Total time:  {row[0]+row[1]} seconds ({(row[0]+row[1])//60} min)")
print()
print("💡 This SQL query works for any number of pause/resume cycles!")
conn.close()

## 📱 Offline-First: Why Track Locally?

A key Strava design insight: **track GPS data on the device, not the server**.

### Why?

1. **Offline support** — runners go through tunnels, rural areas, airplane mode
2. **Bandwidth savings** — instead of sending GPS every 5 seconds, send once at the end
3. **Battery savings** — fewer network calls = longer battery life
4. **Lower server load** — 100× fewer requests to the backend

### How it works:

```
┌─────────────────────────────────────┐
│            MOBILE DEVICE            │
│                                     │
│  GPS sensor → In-memory buffer      │
│                  │                   │
│                  ▼ (every ~10 sec)   │
│             Local storage            │
│          (SQLite / CoreData)         │
│                  │                   │
│                  ▼ (activity done)   │
│         Upload all at once ─────────────────▶ Server
│                                     │
└─────────────────────────────────────┘
```

Let's simulate this pattern.

In [ ]:
import random

def simulate_gps_recording(duration_s=60, interval_s=5):
    """
    Simulate recording GPS points on a mobile device.
    Returns a list of (lat, lon, timestamp) tuples.
    """
    # Start near the SF Ferry Building
    lat, lon = 37.7955, -122.3935
    points = []
    base_time = datetime.now()

    for t in range(0, duration_s, interval_s):
        # Simulate moving south-east along the Embarcadero
        lat -= random.uniform(0.0003, 0.0008)
        lon += random.uniform(0.0003, 0.0008)
        ts = base_time + timedelta(seconds=t)
        points.append((lat, lon, ts))

    return points

# Step 1: Record locally (simulated in-memory buffer)
local_buffer = simulate_gps_recording(duration_s=60, interval_s=5)

print(f"📱 Recorded {len(local_buffer)} GPS points locally")
print(f"   First: ({local_buffer[0][0]:.4f}, {local_buffer[0][1]:.4f})")
print(f"   Last:  ({local_buffer[-1][0]:.4f}, {local_buffer[-1][1]:.4f})")

# Calculate distance on-device using Haversine
on_device_distance = 0
for i in range(len(local_buffer) - 1):
    on_device_distance += haversine(
        local_buffer[i][0], local_buffer[i][1],
        local_buffer[i+1][0], local_buffer[i+1][1]
    )
print(f"   Distance (calculated on device): {on_device_distance:.0f} m")
print()
print("💡 All of this happened without any network calls!")

In [ ]:
# Step 2: Activity complete — batch upload to the server
conn = get_db()
cur = conn.cursor()

# Create a new activity
cur.execute("""
    INSERT INTO activities (user_id, type, state, title, distance_m, duration_s,
                            started_at, completed_at)
    VALUES (1, 'RUN', 'COMPLETE', 'Simulated Offline Run', %s, 60,
            NOW() - INTERVAL '1 minute', NOW())
    RETURNING id;
""", (on_device_distance,))
new_activity_id = cur.fetchone()[0]

# Batch insert all route points in one go
values = []
for seq, (lat, lon, ts) in enumerate(local_buffer, start=1):
    values.append(cur.mogrify(
        "(%s, %s, %s, %s, 0, %s, ST_Point(%s, %s)::geography)",
        (new_activity_id, seq, lat, lon, ts, lon, lat)
    ).decode())

cur.execute(f"""
    INSERT INTO route_points
        (activity_id, seq, latitude, longitude, elevation_m, recorded_at, geom)
    VALUES {','.join(values)};
""")

print(f"✅ Uploaded activity #{new_activity_id} with {len(local_buffer)} GPS points")
print(f"   This was a single batch upload — not {len(local_buffer)} separate requests!")
print()

# Verify with PostGIS distance
cur.execute("""
    WITH ordered AS (
        SELECT geom, LAG(geom) OVER (ORDER BY seq) AS prev_geom
        FROM route_points WHERE activity_id = %s
    )
    SELECT ROUND(SUM(ST_Distance(geom, prev_geom))::numeric, 1)
    FROM ordered WHERE prev_geom IS NOT NULL;
""", (new_activity_id,))
server_dist = cur.fetchone()[0]
print(f"   Server recalculated distance: {server_dist} m")
print(f"   Device calculated distance:   {on_device_distance:.1f} m")

conn.close()

## 📊 Storage Estimation

Let's work through the math that interviewers love to ask about.

**Assumptions** (from the Hello Interview breakdown):
- 100M DAU, each doing 1 activity/day
- Average activity = 30 min
- GPS point every ~3 seconds → 600 points per activity

In [ ]:
# Storage estimation
dau = 100_000_000
activities_per_day = 1
points_per_activity = 600

# Size per activity row (metadata)
metadata_bytes = 100  # id, user_id, type, state, timestamps, distance, etc.

# Size per route point: lat(8) + lon(8) + timestamp(8) + overhead ≈ 24 bytes
bytes_per_point = 24
route_bytes = points_per_activity * bytes_per_point  # ~14.4 KB per activity

total_per_activity = metadata_bytes + route_bytes
total_per_day = dau * total_per_activity
total_per_year = total_per_day * 365

print("📊 Storage Estimation")
print("=" * 50)
print(f"  Per activity: {total_per_activity:,} bytes ({total_per_activity/1024:.1f} KB)")
print(f"  Per day:      {total_per_day/1e12:.1f} TB")
print(f"  Per year:     {total_per_year/1e12:.1f} TB")
print()
print("💡 ~530 TB/year is large but manageable.")
print("   Solutions: shard by time, data tiering (hot/warm/cold), compression.")
print()
print("   Hot  (< 3 months):  fast SSD storage")
print("   Warm (3-12 months): cheaper storage")
print("   Cold (> 1 year):    archival (e.g., S3)")

## 🧹 Cleanup

In [ ]:
# Remove the simulated activity we created
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM route_points WHERE activity_id = %s", (new_activity_id,))
cur.execute("DELETE FROM activities WHERE id = %s", (new_activity_id,))
conn.close()
print(f"🧹 Cleaned up simulated activity #{new_activity_id}")

## 📚 Summary

### Key Takeaways

1. **GPS tracking** records (lat, lon, timestamp) every few seconds
2. **Haversine formula** calculates distance on a sphere — essential for fitness apps
3. **PostGIS** brings spatial math into SQL — indexed, fast, and expressive
4. **Pause/resume** needs a state log to calculate accurate active time
5. **Offline-first** is the key insight: track on-device, upload on completion
6. At ~530 TB/year, sharding + tiering keeps storage manageable

### Next Up

In **Notebook 2**, we'll build the **activity feed and social features** —
showing friends' activities, fan-out queries, and Redis caching for feeds.